In [1]:
import rosalia as rs
import pandas as pd
from astroquery.mpc import MPC

obs = MPC.get_observatory_codes()
print(obs[obs["Name"] == "Euclid"])
print(obs[obs["Name"] == "Hubble Space Telescope"])
print(obs[obs["Name"] == "James Webb Space Telescope"])
print(obs[obs["Name"] == "Nancy Grace Roman Space Telescope"])


Code Longitude cos sin  Name 
---- --------- --- --- ------
 273        --  --  -- Euclid
Code Longitude cos sin          Name         
---- --------- --- --- ----------------------
 250        --  --  -- Hubble Space Telescope
Code Longitude cos sin            Name           
---- --------- --- --- --------------------------
 274        --  --  -- James Webb Space Telescope
Code Longitude cos sin                Name              
---- --------- --- --- ---------------------------------
 289        --  --  -- Nancy Grace Roman Space Telescope


In [2]:
rs.__file__

'/Users/aborlaff/NASA/ROSALIA/rosalia/__init__.py'

In [3]:
import ephessos as ep
print(ep.__file__)

/Users/aborlaff/NASA/EPHESSOS/ephessos/__init__.py


In [6]:
# Telescope definition object 
# telescope = {"pointing": [RA_TARG, DEC_TARG], "PA_Y": PA_Y, "detector_shape": [NAXIS1, NAXIS2], "pixscale": pixscale,
#              "R_mirror": R_mirror, "OBSLOC": OBSLOC, "EXPSTART": EXPSTART, "EXPTIME": EXPTIME}

class exposure():
    
    def __init__(self, filename=None, telescope=None):
        import numpy as np 
        from astropy.time import Time
        import astropy.units as u

        if telescope is not None: 
            
            self.TELESCOP = telescope['TELESCOP']
            self.DATA_SHAPE = telescope['DATA_SHAPE']
            self.RA_TARG = telescope["pointing"][0]
            self.DEC_TARG = telescope["pointing"][1]
            self.PA = telescope['PA_Y']
            self.EXPSTART = telescope['EXPSTART']
            self.EXPTIME = telescope['EXPTIME']
            self.EXPMID = (Time(self.EXPSTART, format="mjd") + self.EXPTIME*u.s/2).mjd
            self.EXPEND = (Time(self.EXPSTART, format="mjd") + self.EXPTIME*u.s).mjd
            # self.PHYSPIX = telescope['PHYSPIX']
            self.PIXSCALE = telescope['PIXSCALE']
            # self.EXPSTART_ISOT = 
            self.MPC_OBSLOC = self.get_mpc_observer_location()
            self.JPL_OBSLOC = self.get_jpl_observer_location()
            self.FILTER_IDENTITY = rs.telescopes.find_filter_in_svo(wavelength=telescope["FILTER_PARAMS"]["NAME"],
                                                                    telescope=telescope["FILTER_PARAMS"]["TELESCOPE"],
                                                                    instrument=telescope["FILTER_PARAMS"]["INSTRUMENT"],
                                                                    detector=telescope["FILTER_PARAMS"]["DETECTOR"], verbose=False)
            # self.XYZ_HELIO_POS = exposure_identity['XYZ_HELIO_POS']
            self.SCIEXTS = [0]

            header = rs.utils.create_custom_wcs(crpix=np.array(self.DATA_SHAPE)/2, 
                                                         crval=[self.RA_TARG, self.DEC_TARG], 
                                                         cdelt=[-self.PIXSCALE,self.PIXSCALE], 
                                                         crota=[-self.PA,-self.PA], 
                                                         projection="TAN")
            
            header["NAXIS1"] = self.DATA_SHAPE[0]
            header["NAXIS2"] = self.DATA_SHAPE[1]

            from astropy.wcs import WCS
            self.ASTROPYWCS = [WCS(header)]
            self.FPA_NEAR_RADIUS = self.get_max_angular_size()
            import os 
            self.FILENAME = os.getcwd() + "/" + self.TELESCOP + "_RA_" + '{:07.3f}'.format(self.RA_TARG) +\
                                     "_DEC_" + '{:07.3f}'.format(self.DEC_TARG) +\
                                     "_MJD_" + '{:07.5f}'.format(self.EXPSTART) +\
                                     "_PA_" + '{:06.2f}'.format(self.PA) + ".fits"

        if filename is not None:
            exposure_identity = rs.utils.exposure_inspector(filename, lite=True)
        
            self.FILENAME = exposure_identity['FILENAME']
            self.TELESCOP = exposure_identity['TELESCOP']
            self.INSTRUME = exposure_identity['INSTRUME']
            self.DETECTOR = exposure_identity['DETECTOR']
            self.RA_TARG = exposure_identity['RA_TARG']
            self.DEC_TARG = exposure_identity['DEC_TARG']
            self.EXPSTART = exposure_identity['EXPSTART']
            self.EXPTIME = exposure_identity['EXPTIME']
            self.EXPMID = (Time(self.EXPSTART, format="mjd") + self.EXPTIME*u.s/2).mjd
            self.EXPEND = (Time(self.EXPSTART, format="mjd") + self.EXPTIME*u.s).mjd
            self.BUNIT = exposure_identity['BUNIT']
            self.EXPSTART_ISOT = exposure_identity['EXPSTART_ISOT']
            self.PA = exposure_identity['PA']
            # self.SCA = exposure_identity['SCA']
            self.HST_TYPE = exposure_identity['HST_TYPE']
            self.FILTER = exposure_identity['FILTER']
            self.FILTER_IDENTITY = exposure_identity['FILTER_IDENTITY']
            # self.PHYSPIX = exposure_identity['PHYSPIX']
            self.PIXSCALE = exposure_identity['PIXSCALE']
            self.SCIEXTS = exposure_identity['SCIEXTS']
            self.DATA_SHAPE = exposure_identity['DATA_SHAPE']
            self.ASTROPYWCS = exposure_identity['ASTROPYWCS']
            self.FILETYPE = exposure_identity['FILETYPE']
            self.MPC_OBSLOC = self.get_mpc_observer_location()
            self.JPL_OBSLOC = self.get_jpl_observer_location()
            self.XYZ_HELIO_POS = exposure_identity['XYZ_HELIO_POS']
            self.FPA_NEAR_RADIUS = self.get_max_angular_size()

    def get_jpl_observer_location(self):
        if self.TELESCOP == "HST" or self.TELESCOP == "Hubble": return("500@-48")
        if self.TELESCOP == "RST" or self.TELESCOP == "Roman": return("500@-211")
        if self.TELESCOP == "Euclid": return("500@-680")

    def get_mpc_observer_location(self):
        if self.TELESCOP == "HST" or self.TELESCOP == "Hubble": return("250")
        if self.TELESCOP == "RST" or self.TELESCOP == "Roman": return("289")
        if self.TELESCOP == "Euclid": return("273")


    def get_detector_corners(self):
        detector_corners = []
        for i in range(len(self.SCIEXTS)):
            detector_corners.append(rs.detectors.get_detector_corners(self.ASTROPYWCS[i]))
        return(detector_corners)
    
    def get_max_angular_size(self):
        return(rs.utils.find_max_angular_size_of_image(wcs=self.ASTROPYWCS, ra_cen=self.RA_TARG, dec_cen=self.DEC_TARG))
    

    def plot_footprint(self, ax=None, color='red', label=None, **kwargs):
        import numpy as np
        import matplotlib.pyplot as plt
        
        ra_dec_constraints = rs.gaia.find_ra_dec_constraints(self.RA_TARG, self.DEC_TARG, radius=self.FPA_NEAR_RADIUS, verbose=False)
        exp_corners = self.get_detector_corners()

        if ax is None:
            fig, ax = plt.subplots(figsize=(8,8))
        for i in range(len(self.SCIEXTS)):
            x = np.array(exp_corners[i]["corners_world"][:,0].tolist() + [exp_corners[i]["corners_world"][0,0]])
            y = np.array(exp_corners[i]["corners_world"][:,1].tolist() + [exp_corners[i]["corners_world"][0,1]])
            ax.plot(x, y, color=color, label=label, **kwargs)
            ax.set_xlim(ra_dec_constraints["ra_max"], ra_dec_constraints["ra_min"])
            ax.set_ylim(ra_dec_constraints["dec_min"], ra_dec_constraints["dec_max"])
            ax.set_xlabel("Right Ascension (degree)")
            ax.set_ylabel("Declination (degree)")
        return(ax)
    
    def find_nearby_ssos(self):
        import ephessos as ep 
        sso_cone_search = ep.core.cone_search(ra=self.RA_TARG, dec=self.DEC_TARG, mjd=self.EXPSTART, search_radius=self.FPA_NEAR_RADIUS, verbose=False)
        return(sso_cone_search)

    def get_nearby_sources(self, g_mag_max=15, verbose=False):
        hybrid_catalog = rs.psf.get_hybrid_catalog(ra=self.RA_TARG, dec=self.DEC_TARG,
                                                   radius=1,
                                                   lambda_ref=self.FILTER_IDENTITY["filter_lambda_ref"],
                                                   MJD=self.EXPSTART,
                                                   observer=self.TELESCOP,
                                                   g_mag_max = g_mag_max,
                                                   verbose=verbose,
                                                   query_filename=self.FILENAME.replace(".fits", ".csv"))
        return(hybrid_catalog)
        

    def get_nearby_ssos(self, verbose=True, time_step="30s"):
        import ephessos as ep
        cone_search = ep.core.cone_search(ra=self.RA_TARG, dec=self.DEC_TARG, mjd=self.EXPSTART, 
                                          search_radius=self.FPA_NEAR_RADIUS*60*60, observatory=self.MPC_OBSLOC, verbose=verbose)
        ephessos_df = ep.core.ephessos(sso_search=cone_search, mjd_start=self.EXPSTART, mjd_end=self.EXPEND, obs_center=self.JPL_OBSLOC, step_size=time_step, verbose=verbose)
        return(ephessos_df)
    
    



In [7]:
# filename="/Users/aborlaff/NASA/AURELIEN/UFO/MAST_2022-05-05T1554/HST/ic1601hsq/ic1601hsq_flt.fits"
# hst_exp = exposure(filename="/Users/aborlaff/NASA/AURELIEN/UFO/MAST_2022-05-05T1554/HST/ic1601hsq/ic1601hsq_flt.fits")
filename = "/Users/aborlaff/Downloads/MAST_2026-05-08T2317/HST/j90za9qmq/j90za9qmq_flc.fits"
hst_exp = exposure(filename=filename)
hst_exp.FPA_NEAR_RADIUS = hst_exp.FPA_NEAR_RADIUS*5
print(hst_exp.FPA_NEAR_RADIUS)
ssos = hst_exp.get_nearby_ssos(time_step="1m")

TypeError: 'NoneType' object is not subscriptable

In [ ]:
ssos[0]

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm

hst_exp.plot_footprint()

for i in tqdm(range(len(ssos))):
    print(ssos[i]["Designation"][0])
    plt.plot(ssos[i]["RA_deg_ICRF"], ssos[i]["DEC_deg_ICRF"])

In [ ]:
for i in range(len(ssos)):
    print(str(ssos[i]["RA_deg_ICRF"][0]) + " " + str(ssos[i]["DEC_deg_ICRF"][0]))

In [ ]:
from astropy.time import Time
#t1 = Time('2006-01-01', scale="tt")
t1 = Time(hst_exp.EXPSTART, format="mjd", scale="tt")
t2 = Time(hst_exp.EXPEND, format="mjd", scale="tt")
#t2 = Time('2006-01-02', scale="tt")

print(t1.isot)
print(t2.isot)
# print(t.td)

In [ ]:
https://ssd.jpl.nasa.gov/api/horizons.api?format=text&COMMAND='499'&OBJ_DATA='YES'&MAKE_EPHEM='YES'&EPHEM_TYPE='OBSERVER'&CENTER='500@399'&START_TIME='2006-01-01'&STOP_TIME='2006-01-20'&STEP_SIZE='1%20d'&QUANTITIES='1,9,20,23,24,29'

In [ ]:
ssos

In [ ]:
import pympc
pympc.utils.ensure_obs_codes_cached(update=True)

In [ ]:
custom_exposure.get_detector_corners()


In [ ]:
nearby_catalog = custom_exposure.get_nearby_sources()

In [ ]:
# Zodiacal light STScI
# roman_notebooks/notebooks/background_visualization_tool/rbt.py     def read_bkg_data_from_url(self, cache_file, verbose=False):

In [ ]:


TELESCOP="STARFOX"
RA_TARG = 100
DEC_TARG = 23
PA_Y = 32
NAXIS1=2048
NAXIS2=4096
PIXSCALE=8/60/60
R_mirror=1
OBSLOC=273
EXPSTART=61000
EXPTIME=10
FILTER_PARAMS={"NAME": "F606W", "TELESCOPE": "HST", "INSTRUMENT": "ACS", "DETECTOR": "WFC"}

telescope = {"TELESCOP":TELESCOP, "pointing": [RA_TARG, DEC_TARG], "PA_Y": PA_Y, "DATA_SHAPE": [NAXIS1, NAXIS2], "PIXSCALE": PIXSCALE,
             "R_mirror": R_mirror, "FILTER_PARAMS":FILTER_PARAMS, "OBSLOC": OBSLOC, "EXPSTART": EXPSTART, "EXPTIME": EXPTIME}

custom_exposure = exposure(telescope=telescope)
custom_exposure.get_detector_corners()
custom_exposure.plot_footprint()
custom_exposure.ASTROPYWCS[0].proj_plane_pixel_scales()

In [ ]:
import glob
roman_fits = glob.glob("/Users/aborlaff/NASA/ROSALIA_DEVEL/notebooks/CAR/WFI*[0-9].fits")

for i in range(len(roman_fits)):
    print(roman_fits[i])
    roman_exposure = exposure(roman_fits[i])
    roman_exposure.plot_footprint()
    roman_exposure.find_nearby_ssos()

In [ ]:
(/60/60*4096)**2

In [ ]:
# roman_exposure.get_detector_corners()
rs.utils.find_max_angular_size_of_image(wcs=roman_exposure.ASTROPYWCS)    

In [ ]:
ep.core.get_last_mpcorb()

In [ ]:
exp_corners[0]["corners_world"][:,0].tolist()

In [ ]:
import pympc 
pympc.update_catalogue()

In [ ]:
from astropy.coordinates import SkyCoord

from astropy.time import Time

from astroquery.jplhorizons import Horizons
import numpy as np

epoch_start = Time('2026-04-02 01:58:32.3051', scale="tt")
epoch_end   = Time('2026-04-10 23:54:22.857', scale="tt")
epochs = {'start': epoch_start.isot, 'stop': epoch_end.isot, 'step': "1m"}

q = Horizons('2026-069A', location='@399', epochs=epochs)

tab = q.vectors(refplane='earth')
import matplotlib.pyplot as plt
plt.plot(tab["x"], tab["y"])

In [ ]:
from astropy.coordinates import SkyCoord

from astropy.time import Time

from astroquery.jplhorizons import Horizons
import numpy as np

epoch_start = Time('2006-04-02 01:58:32.3051', scale="tt")
epoch_end   = Time('2006-04-10 23:54:22.857', scale="tt")
epochs = {'start': epoch_start.isot, 'stop': epoch_end.isot, 'step': "1m"}

q = Horizons('Hubble', location='@399', epochs=epochs)

tab = q.vectors(refplane='earth')
import matplotlib.pyplot as plt
plt.plot(tab["x"], tab["y"])